In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.

if customer inquires a ticket price for a city, do the following steps:
1. check the existing data through get ticket price function
2. if not found, call the second tool which is setting a new ticket data
3. first check the city in the ticket_price list
4. if found, get the price and insert the new city and price into the database
5. then return the new price to the customer. 
6. if not found in the ticket price list, say so.
"""

In [ ]:
import sqlite3

In [ ]:

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()
    

In [ ]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
# Pre set ticket data
existing_data = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in existing_data.items():
    set_ticket_price(city, price)

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"



In [ ]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}


In [ ]:
set_price_function = {
  "name": "set_ticket_price",
  "description": "Create or update a ticket to a desired city.",
  "parameters": {
    "type": "object",
    "properties": {
      "destination_city": {
        "type": "string",
        "description": "The city that the customer wants to travel to "
      },
    },
    "required": ["destination_city"],
    "additionalProperties": False
  }
}


In [ ]:
tools = [{"type": "function", "function": price_function},
         {"type": "function", "function": set_price_function}
]

In [ ]:
#-Test

tools

In [ ]:
def handle_tool_calls(message):
    responses = []
    what_content = ""
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city').lower()        

        if tool_call.function.name == "get_ticket_price":
            price_details = get_ticket_price(city)
            what_content = price_details
            
        elif tool_call.function.name == "set_ticket_price":
            print('set func() called')

            if city in ticket_prices:
                price = ticket_prices[city]                                
                set_ticket_price(city,price)
                what_content =  f"Ticket price to {city} is ${price}"
                print('set func() called-- YES')
            else:
                what_content = "No city found!"      
                print('set func() -- NO')
        else:
            what_content = "Unknown tool"

        responses.append({
            "role": "tool",
            "content": what_content,
            "tool_call_id": tool_call.id
        })   
    return responses

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":    # Repeat if OpenAI requests another tool.
        message = response.choices[0].message
        tool_resps = handle_tool_calls(message)
        messages.append(message)
        messages.extend(tool_resps)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
# New data to be added
ticket_prices = {"berlin":500, "chicago": 450, "rome": 1000, "seoul": 1500}

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()